# 09. 토지피복(02) → 격자 표면조성 — 표면축 물리화

놓쳤던 02 토지피복도(144,677폴리곤)로 격자별 실제 표면 비율 산정.
blockType(조잡) 대체·보강. UTCI 공간판별·냉각(수역/나무)·계절역전 검증의 토대.

- L1 → 5클래스: 불투수(imperv)·나무(tree,산림)·저식생(lowveg,초지+농업)·수역(water)·나지(bare)
- 격자별 면적비율. `grid_features`에 추가.

산출물: grid_features(+imperv_frac·tree_frac·lowveg_frac·water_frac·bare_frac), fig12

In [1]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))
import numpy as np, pandas as pd, geopandas as gpd
from src.config import DATA_RAW, DATA_PROCESSED, OUT_FIGURES, CRS_LANDSAT, CRS_SUBMIT

CLASS = {"100": "imperv", "300": "tree", "400": "lowveg", "200": "lowveg",
         "600": "water", "500": "water", "700": "bare"}
lc = gpd.read_file(DATA_RAW / "02._토지피복도.geojson")[["L1_CODE", "geometry"]]
lc["cls"] = lc["L1_CODE"].astype(str).map(CLASS).fillna("imperv")
lc = lc.to_crs(CRS_LANDSAT)
print("토지피복", len(lc), "폴리곤. 클래스:", lc["cls"].value_counts().to_dict())

토지피복 144677 폴리곤. 클래스: {'imperv': 89970, 'lowveg': 42591, 'water': 7984, 'tree': 3520, 'bare': 612}


In [2]:
# 격자 × 토지피복 교차면적 → 클래스별 비율
grid = gpd.read_file(DATA_PROCESSED / "grid_features.geojson")
grid = grid.drop(columns=[c for c in grid.columns if c.endswith("_frac")], errors="ignore")
gL = grid.to_crs(CRS_LANDSAT)

pairs = gpd.sjoin(gL[["grid_id", "geometry"]], lc, predicate="intersects").reset_index(drop=True)
gl = gL.set_index("grid_id").geometry.loc[pairs["grid_id"].values].reset_index(drop=True)
ll = lc.geometry.loc[pairs["index_right"].values].reset_index(drop=True)
pairs["iarea"] = gl.intersection(ll, align=False).area
tab = pairs.groupby(["grid_id", "cls"])["iarea"].sum().unstack(fill_value=0)
tab = tab.div(tab.sum(axis=1).clip(lower=1), axis=0)          # 비율 정규화
tab.columns = [f"{c}_frac" for c in tab.columns]
grid = grid.merge(tab, on="grid_id", how="left").fillna({c: 0 for c in tab.columns})
print("지구별 표면조성(평균):")
print(grid.groupby("zoneName")[[c for c in grid.columns if c.endswith('_frac')]].mean().round(2))

지구별 표면조성(평균):
               bare_frac  imperv_frac  lowveg_frac  tree_frac  water_frac
zoneName                                                                 
고양창릉 공공주택지구         0.01         0.14         0.43       0.28        0.13
남양주왕숙 공공주택지구        0.02         0.30         0.51       0.07        0.10
성남분당                0.02         0.60         0.21       0.12        0.05
성남판교지구 택지개발사업       0.01         0.50         0.26       0.21        0.03


In [3]:
grid.to_crs(CRS_SUBMIT).to_file(DATA_PROCESSED / "grid_features.geojson", driver="GeoJSON", COORDINATE_PRECISION=6)

import matplotlib.pyplot as plt
from src.viz.fonts import setup_font
setup_font()   # 한글 폰트 자동 적용(COMPAS 미보유 시 동봉 폰트 등록)
fig, ax = plt.subplots(1, 3, figsize=(20, 7))
for a, col, ttl, cmap in [(ax[0], "imperv_frac", "불투수 비율", "pink_r"),
                          (ax[1], "tree_frac", "산림 비율", "Greens"),
                          (ax[2], "water_frac", "수역 비율", "Blues")]:
    grid.plot(column=col, cmap=cmap, ax=a, linewidth=0, legend=True, legend_kwds={"shrink": 0.5}, vmin=0, vmax=1)
    a.set_title(ttl); a.set_axis_off()
plt.tight_layout(); plt.savefig(OUT_FIGURES / "fig12_landcover.png", dpi=120); plt.close()